<a href="https://colab.research.google.com/github/PKpacheco/linear-algebra-for-machine-learning/blob/main/01_linear-classifier-fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Algebra for Machine Learning

This notebook explores how linear algebra concepts can be applied to equipment condition data.

The dataset contains measurements for voltage, current, and temperature. Each observation is classified as either `normal` or `failed`.

The project covers:

* Data inspection
* Mean and standard deviation
* Feature standardization
* Vectors and vector magnitude
* Dot products
* Linear classification
* Loss functions and accuracy


In [3]:
# import

# Pandas is used to load and manipulate tabular data.
import pandas as pd

# NumPy provides mathematical operations for vectors and matrices.
import numpy as np

# Plotly can be used to create interactive data visualizations.
import plotly.express as px

from math import sqrt

Understanding the Dataset

Before performing mathematical calculations, it is important to inspect the structure and quality of the data.

The dataset contains:

condition: the known equipment condition
voltage: the measured voltage
current: the measured electrical current
temperature: the measured temperature

In [4]:
#loading dataset
DATA_URL = (
    "https://raw.githubusercontent.com/PKpacheco/linear-algebra-for-machine-learning/refs/heads/main/data_algebra.csv"
)

csv_data = pd.read_csv(DATA_URL)
csv_data.head()

,condition,voltage,current,temperature
0,normal,13.036111,458.460370,6.879090
1,normal,11.135509,458.156781,6.394905
2,normal,10.296578,459.431151,2.736525
3,normal,10.553064,457.141002,5.624983
4,normal,9.880487,460.601899,1.843793


The dataset contains 10 observations and 4 columns.

The `condition` column contains text labels, while voltage, current, and temperature are numerical features.

All columns contain 10 non-null values, which means there are no missing values in this dataset.


In [5]:
#info data
csv_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   condition    10 non-null     object 
 1   voltage      10 non-null     float64
 2   current      10 non-null     float64
 3   temperature  10 non-null     float64
dtypes: float64(3), object(1)
memory usage: 452.0+ bytes


In [6]:
# Return the number of rows and columns in the dataset.
rows, columns = csv_data.shape

print("Number of observations:", rows)
print("Number of columns:", columns)

Number of observations: 10
Number of columns: 4


In [7]:
# Count missing values in each column.
csv_data.isna().sum()

,0
condition,0
voltage,0
current,0
temperature,0


In [8]:
#calculate mean

mean_voltage = csv_data['voltage'].mean().round(2)
mean_current = csv_data['current'].mean().round(2)
mean_temperature = csv_data['temperature'].mean().round(2)

print('>>> mean_voltage = ', mean_voltage)
print('>>> mean_current = ', mean_current)
print('>>> mean_temperature =',mean_temperature)

>>> mean_voltage =  19.31
>>> mean_current =  2530.08
>>> mean_temperature = 28.38


In [9]:
# Calculate the standard deviation of each numerical feature.
std_voltage = round(csv_data["voltage"].std(), 2)
std_current = round(csv_data["current"].std(), 2)
std_temperature = round(csv_data["temperature"].std(), 2)

print("Standard deviation - voltage:", std_voltage)
print("Standard deviation - current:", std_current)
print("Standard deviation - temperature:", std_temperature)

print('>>> std_voltage =', std_voltage)
print('>>> std_current =', std_current)
print('>>> std_temperature =',std_temperature)

Standard deviation - voltage: 9.24
Standard deviation - current: 2183.37
Standard deviation - temperature: 25.16
>>> std_voltage = 9.24
>>> std_current = 2183.37
>>> std_temperature = 25.16


In [10]:
#standardized voltage

csv_data['standardized_voltage'] = ((csv_data['voltage'] - mean_voltage )/ std_voltage).round(2)
csv_data.head()

,condition,voltage,current,temperature,standardized_voltage
0,normal,13.036111,458.460370,6.879090,-0.68
1,normal,11.135509,458.156781,6.394905,-0.88
2,normal,10.296578,459.431151,2.736525,-0.98
3,normal,10.553064,457.141002,5.624983,-0.95
4,normal,9.880487,460.601899,1.843793,-1.02


In [11]:
#standardized current
csv_data['standardized_current'] = ((csv_data['current'] - mean_current )/ std_current).round(2)
csv_data.head()

,condition,voltage,current,temperature,standardized_voltage,standardized_current
0,normal,13.036111,458.460370,6.879090,-0.68,-0.95
1,normal,11.135509,458.156781,6.394905,-0.88,-0.95
2,normal,10.296578,459.431151,2.736525,-0.98,-0.95
3,normal,10.553064,457.141002,5.624983,-0.95,-0.95
4,normal,9.880487,460.601899,1.843793,-1.02,-0.95


In [12]:
#standardized temperature
csv_data['standardized_temperature'] = ((csv_data['temperature'] - mean_temperature )/ std_temperature).round(2)
csv_data.head()

,condition,voltage,current,temperature,standardized_voltage,standardized_current,standardized_temperature
0,normal,13.036111,458.460370,6.879090,-0.68,-0.95,-0.85
1,normal,11.135509,458.156781,6.394905,-0.88,-0.95,-0.87
2,normal,10.296578,459.431151,2.736525,-0.98,-0.95,-1.02
3,normal,10.553064,457.141002,5.624983,-0.95,-0.95,-0.90
4,normal,9.880487,460.601899,1.843793,-1.02,-0.95,-1.05


## Preparing the Data for Classification

The standardized voltage, current, and temperature values will be used as
input features for a binary classifier.

The equipment condition is converted into a numerical target:

- `normal` = 0
- `failed` = 1

In [29]:
# Select the standardized features used by the classifier.
feature_columns = [
    "standardized_voltage",
    "standardized_current",
    "standardized_temperature",
]

# Create the feature matrix.
# Each row represents one equipment observation.
X = csv_data[feature_columns].to_numpy()

# Convert the condition labels into numerical values.
# normal = 0 and failed = 1.
y = csv_data["condition"].map({
    "normal": 0,
    "failed": 1,
}).to_numpy()

print("Feature matrix shape:", X.shape)
print("Target values:", y)

Feature matrix shape: (10, 3)
Target values: [0 0 0 0 0 1 1 1 1 1]


In [30]:
# Display the standardized features and their numerical target.
prepared_data = csv_data[
    [
        "condition",
        "standardized_voltage",
        "standardized_current",
        "standardized_temperature",
    ]
].copy()

prepared_data["target"] = y

prepared_data.round(2)

,condition,standardized_voltage,standardized_current,standardized_temperature,target
0,normal,-0.68,-0.95,-0.85,0
1,normal,-0.88,-0.95,-0.87,0
2,normal,-0.98,-0.95,-1.02,0
3,normal,-0.95,-0.95,-0.90,0
4,normal,-1.02,-0.95,-1.05,0
5,failed,0.90,0.95,1.12,1
6,failed,0.99,0.95,1.07,1
7,failed,1.59,0.94,0.81,1
8,failed,0.60,0.95,0.74,1
9,failed,0.44,0.94,0.98,1


## Training a Logistic Regression Classifier

Logistic regression is a linear classification algorithm.

Instead of manually assigning weights and a bias, the model learns these
parameters from the relationship between the standardized measurements and
the known equipment conditions.

Because this dataset contains only ten observations, this training is used
only to demonstrate how the algorithm works.

In [31]:
from sklearn.linear_model import LogisticRegression

# Create the logistic regression classifier.
classifier = LogisticRegression()

# Train the model using the standardized features and known conditions.
classifier.fit(X, y)

# Extract the weights learned for each feature.
learned_weights = classifier.coef_[0]

# Extract the learned bias.
learned_bias = classifier.intercept_[0]

print("Learned weights:")
print("Voltage:", round(learned_weights[0], 4))
print("Current:", round(learned_weights[1], 4))
print("Temperature:", round(learned_weights[2], 4))
print("Bias:", round(learned_bias, 4))

Learned weights:
Voltage: 0.7727
Current: 0.8679
Temperature: 0.8499
Bias: 0.0138


## Interpreting the Learned Parameters

The logistic regression model learned one weight for each standardized feature.

Because `failed` was encoded as 1, a positive weight means that an increase in
the feature increases the model's score toward the failed class.

In this model, current received the largest weight, followed by temperature
and voltage. However, the dataset contains only ten observations, so these
weights should not be interpreted as reliable real-world feature importance.


Learned weights:
Voltage: 0.7727
Current: 0.8679
Temperature: 0.8499
Bias: 0.0138

In [32]:
# Calculate the linear score for each observation.
# Formula: score = features · weights + bias
linear_scores = X @ learned_weights + learned_bias

# Store the scores in the DataFrame.
csv_data["linear_score"] = linear_scores

csv_data[
    [
        "condition",
        "linear_score",
    ]
].round(4)

,condition,linear_score
0,normal,-2.0586
1,normal,-2.2301
2,normal,-2.4349
3,normal,-2.3097
4,normal,-2.4913
5,failed,2.4856
6,failed,2.5126
7,failed,2.7466
8,failed,1.9308
9,failed,2.0025


In [34]:
# Define the sigmoid function.
def sigmoid(value):
    return 1 / (1 + np.exp(-value))


# Convert the linear scores into failed-class probabilities.
manual_probabilities = sigmoid(linear_scores)

# Store the manually calculated probabilities.
csv_data["manual_failed_probability"] = manual_probabilities

csv_data[
    [
        "condition",
        "linear_score",
        "manual_failed_probability",
    ]
].round(4)

,condition,linear_score,manual_failed_probability
0,normal,-2.0586,0.1132
1,normal,-2.2301,0.0971
2,normal,-2.4349,0.0806
3,normal,-2.3097,0.0903
4,normal,-2.4913,0.0765
5,failed,2.4856,0.9231
6,failed,2.5126,0.9250
7,failed,2.7466,0.9397
8,failed,1.9308,0.8733
9,failed,2.0025,0.8811


In [35]:
# Calculate probabilities using the trained Scikit-learn model.
sklearn_probabilities = classifier.predict_proba(X)[:, 1]

# Compare the manual calculation with the library result.
probability_comparison = pd.DataFrame({
    "manual_probability": manual_probabilities,
    "sklearn_probability": sklearn_probabilities,
})

probability_comparison.round(6)

,manual_probability,sklearn_probability
0,0.113187,0.113187
1,0.097077,0.097077
2,0.080551,0.080551
3,0.090321,0.090321
4,0.076471,0.076471
5,0.923124,0.923124
6,0.925021,0.925021
7,0.939720,0.939720
8,0.873336,0.873336
9,0.881054,0.881054


In [36]:
# Convert probabilities into binary predictions.
# Probabilities of 0.5 or greater are classified as failed.
manual_predictions = (
    manual_probabilities >= 0.5
).astype(int)

# Convert numerical predictions into readable labels.
predicted_conditions = np.where(
    manual_predictions == 1,
    "failed",
    "normal",
)

csv_data["predicted_condition"] = predicted_conditions

csv_data[
    [
        "condition",
        "manual_failed_probability",
        "predicted_condition",
    ]
].round({
    "manual_failed_probability": 4,
})

,condition,manual_failed_probability,predicted_condition
0,normal,0.1132,normal
1,normal,0.0971,normal
2,normal,0.0806,normal
3,normal,0.0903,normal
4,normal,0.0765,normal
5,failed,0.9231,failed
6,failed,0.9250,failed
7,failed,0.9397,failed
8,failed,0.8733,failed
9,failed,0.8811,failed


In [37]:
from sklearn.metrics import accuracy_score

# Compare the predicted classes with the known classes.
training_accuracy = accuracy_score(
    y,
    manual_predictions,
)

print("Training accuracy:", round(training_accuracy, 4))
print("Training accuracy percentage:", f"{training_accuracy:.0%}")

Training accuracy: 1.0
Training accuracy percentage: 100%


This accuracy was calculated using the same observations used to train the
model. It demonstrates that the calculations work, but it is not a reliable
estimate of performance on new data.

In [38]:
from sklearn.metrics import log_loss

# Calculate binary cross-entropy using Scikit-learn.
library_loss = log_loss(
    y,
    manual_probabilities,
)

print(
    "Binary cross-entropy:",
    round(library_loss, 4),
)

Binary cross-entropy: 0.0963


In [39]:
# Prevent logarithm calculations from receiving exactly 0 or 1.
epsilon = 1e-15

safe_probabilities = np.clip(
    manual_probabilities,
    epsilon,
    1 - epsilon,
)

# Calculate binary cross-entropy manually.
manual_loss = -np.mean(
    y * np.log(safe_probabilities)
    + (1 - y) * np.log(1 - safe_probabilities)
)

print(
    "Manual binary cross-entropy:",
    round(manual_loss, 4),
)

Manual binary cross-entropy: 0.0963
